In [8]:
# --- STEP 1: SETUP ---
# Install the necessary libraries
! pip install python-dotenv --upgrade --quiet langchain-groq

import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Securely set your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")



In [9]:
# --- STEP 2: DEFINE EXPERTS ---
# We use Mixtral 8x7B for high capacity and specialized personas
MODEL_CONFIG = {
    "technical": "You are a Senior Technical Support Engineer. Provide rigorous, code-focused, and precise troubleshooting steps. Avoid fluff.",
    "billing": "You are a Billing Specialist. Be empathetic and focus on financial policies and refund procedures. Keep it professional.",
    "general": "You are a friendly Customer Success Manager. Handle general inquiries with a warm and helpful tone.",
    "tool_use": "You are a Data Retrieval Assistant. Your job is to identify when a user is asking for real-time data like cryptocurrency prices."
}


In [10]:
# --- UPDATED STEP 3: THE ROUTER ---
def route_prompt(user_input):
    router_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
    router_prompt = ChatPromptTemplate.from_template(
        "Classify the intent of this customer query into exactly one of these categories: "
        "[technical, billing, general, tool_use]. \n"
        "Use 'tool_use' if the user asks for real-time data like Bitcoin prices.\n"
        "Return ONLY the single word name of the category.\n\n"
        "Query: {query}"
    )
    chain = router_prompt | router_llm | StrOutputParser()
    return chain.invoke({"query": user_input}).lower().strip()

In [11]:
# --- NEW STEP: THE TOOLS ---
def get_mock_bitcoin_price():
    # In a real app, this would call an API like CoinGecko
    return "The current price of Bitcoin is $96,432.10 USD (Mock Data)."

In [12]:
# --- UPDATED STEP 4: THE ORCHESTRATOR ---
def process_request(user_input):
    category = route_prompt(user_input)
    print(f"--- Routing to: {category.upper()} EXPERT ---")

    # INTERCEPT: If it's a tool request, return mock data immediately
    if category == "tool_use":
        if "bitcoin" in user_input.lower():
            return get_mock_bitcoin_price()
        else:
            return "I can currently only fetch Bitcoin prices. Please ask about Bitcoin."

    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])
    expert_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    expert_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])

    chain = expert_prompt | expert_llm | StrOutputParser()
    return chain.invoke({"query": user_input})

In [13]:
print(process_request("What is the current price of Bitcoin?"))

--- Routing to: TOOL_USE EXPERT ---
The current price of Bitcoin is $96,432.10 USD (Mock Data).


In [14]:
# --- TEST IT ---
print(process_request("My python script is throwing an IndexError on line 5."))


--- Routing to: TECHNICAL EXPERT ---
**Error Analysis**

To troubleshoot the `IndexError` on line 5, we need to examine the code surrounding that line. Please provide the code snippet, specifically lines 1-10, to help identify the issue.

**Initial Troubleshooting Steps**

1. **Verify list/index access**: Check if you're trying to access an index that is out of range.
2. **Check list initialization**: Ensure that the list is properly initialized before attempting to access its elements.
3. **Review loop conditions**: If using loops, verify that the loop conditions are correctly set to avoid index out-of-range errors.

**Code Review**

Please provide the code snippet, and I'll help you:

* Identify the specific line causing the error
* Analyze the surrounding code
* Provide a corrected version of the code, if applicable

Example code format:
```python
# your code here
```
Please paste your code, and I'll assist you with the troubleshooting process.
